In [3]:
# Import pandas for reading and manipulating the dataset
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import train_test_split to divide the dataset into training and testing data
from sklearn.model_selection import train_test_split

# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import Tokenizer to convert words into numerical token IDs
from tensorflow.keras.preprocessing.text import Tokenizer

# Import pad_sequences to make all review sequences equal in length
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import Sequential to create the neural network
from tensorflow.keras.models import Sequential

# Import the layers required for the deep learning model
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

# Import matplotlib for plotting graphs
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("data/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
print("Shape:",df.shape)
print("Description:",df.describe())

Shape: (50000, 2)
Description:                                                    review sentiment
count                                               50000     50000
unique                                              49582         2
top     Loved today's show!!! It was a variety and not...  positive
freq                                                    5     25000


In [6]:
# Convert the string sentiment into numerical values
# Positive reviews are assigned the value 1
# Negative reviews are assigned the value 0

df["sentiment"] = df["sentiment"].apply(
    lambda x: 1 if x == "positive" else 0
)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [7]:
# Separate the review column as input data
X = df["review"]

# Separate the sentiment column as target data
Y = df["sentiment"]

# Split the dataset into training and testing sets
# 80% of the data is used for training 
# 20% of the data is used for testing
# stratify = y maintains a similar positive and negative ratio

X_train, X_test, y_train , y_test = train_test_split(
    X,
    Y,
    test_size = 0.20,
    random_state = 42,
    stratify = Y
)

print("\n Training Sample", len(X_train))
print("\n Training Sample", len(X_test))


 Training Sample 40000

 Training Sample 10000


In [17]:
#Define the maximum vocabulary size
#Only the 10,000 most frequent words will be considered
vocab_size = 10000
tokenizer = Tokenizer(
    num_words = vocab_size,
    oov_token = "<OOV>"

 )
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)

X_test_seq = tokenizer.texts_to_sequences(X_test) 


In [18]:
max_length = 200

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [19]:
max_length = 200

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [21]:
#Define the size of each word vector
#Creat deep learning model
embedding_dim = 1000
model = Sequential()
#Embedding 
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length
    )
)
model.add(
    SimpleRNN(64)
)

model.add(
    Dense(1,activation="sigmoid")
)
print("Model Summary:")
model.summary()

Model Summary:


C:\Users\Rajguru\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
#Binary cross-entropy is appropriate for two-class classificatoin
#Accuracy is used as the evaluation metrics

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
#Train the model
#validation_split=0.2 reverse 20% of the tranning data for validation
#epoch control the number of the times the model has seen the training data 
#batch_size determines how many reviews are processed at the time 
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.20
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 48s 93ms/step - accuracy: 0.5022 - loss: 0.6961 - val_accuracy: 0.5174 - val_loss: 0.6912
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 89ms/step - accuracy: 0.5955 - loss: 0.6388 - val_accuracy: 0.5419 - val_loss: 0.7029
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 89ms/step - accuracy: 0.6769 - loss: 0.5098 - val_accuracy: 0.5311 - val_loss: 0.8073
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 44s 89ms/step - accuracy: 0.7055 - loss: 0.4403 - val_accuracy: 0.5480 - val_loss: 0.8902
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - accuracy: 0.7293 - loss: 0.4084 - val_accuracy: 0.5379 - val_loss: 0.9553


In [20]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
)
#Display the test loss and accuracy
print("\nTest Loss:",test_loss)
print("\nTest Accurac:",test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5231 - loss: 0.9927

Test Loss: 0.9926933646202087

Test Accurac: 0.5231000185012817
